# Horizontal Memory Partitioning — **DeepSpeed ZeRO 1–3 vs. PyTorch FSDP**
### Tools & Frameworks  ·  runs on 1 GPU, scales to N without editing the training script

> Every other tools notebook in this folder made a *single* GPU go further: QLoRA shrank the weights, Unsloth's kernels shrank the activations, Axolotl made the run reproducible. This one removes the GPU boundary itself — it partitions **one model's training state across many devices**, so the constraint stops being "what fits in 80 GB" and becomes "what fits in `N × 80` GB."
>
> ⚠️ **Read this before you run anything.** There are **two independent axes** here that almost everyone conflates, and conflating them is why so much distributed-training advice is wrong:
> - **Sharding** divides state across *peers*. It scales with `world_size` and does **nothing at all on one GPU.**
> - **Offload** moves state to *host RAM or NVMe*. It works on one GPU and its benefit is independent of `world_size`.
>
> A single-GPU box can therefore measure the offload axis honestly and the sharding axis not at all. This notebook is built around that fact: it ships an **exact analytical memory model** for the sharding axis (which is arithmetic, not opinion — no cluster required to be correct about it), and a **real, measured benchmark** that auto-scales to whatever GPUs you actually have. On one T4 you get a genuine offload result plus a wiring check; on 8×H100 the same script produces the full matrix.
>
> API claims are checked against `deepspeedai/DeepSpeed@master`, `pytorch/pytorch@main` and `huggingface/accelerate@main`.

---

## 1. Deep-Dive Conceptual Roadmap & Dataset Ecosystem

### What it is (precise terminology)

- **Data parallelism (DDP)** replicates the entire model on every rank and all-reduces gradients each step. Memory per rank is `O(model state)` and **independent of `world_size`** — you buy throughput, never capacity. It is the thing both techniques below are trying to fix.
- **ZeRO — Zero Redundancy Optimizer** (*Rajbhandari et al., 2020*) observes that DDP's replication of optimizer states, gradients and parameters is pure redundancy, and partitions each in turn. Writing `Ψ` for parameter count and `K` for optimizer-state bytes per parameter (`K = 12` for mixed-precision AdamW: fp32 master copy + fp32 momentum + fp32 variance), per-rank memory is:

  | Stage | What is partitioned | Per-rank bytes | 7.5 B params, `P = 64` |
  |---|---|---|---|
  | **DDP / ZeRO-0** | nothing | `(2 + 2 + K)Ψ` | 120 GB |
  | **ZeRO-1** | optimizer states | `2Ψ + 2Ψ + KΨ/P` | 31.4 GB |
  | **ZeRO-2** | + gradients | `2Ψ + (2 + K)Ψ/P` | 16.6 GB |
  | **ZeRO-3** | + parameters | `(2 + 2 + K)Ψ/P` | 1.9 GB |

  ZeRO-3 is where "beyond single-GPU VRAM" actually happens: nothing is fully resident, and a layer's parameters are **all-gathered just before use and freed immediately after.**
- **ZeRO-Offload / ZeRO-Infinity** move the partitioned state to **host RAM** (`offload_optimizer`, `offload_param`) or to **NVMe** (`device: "nvme"`), trading PCIe/SSD bandwidth for VRAM. This is the axis that works on one GPU.
- **FSDP — Fully Sharded Data Parallel** is PyTorch's native implementation of the same idea, descended from FairScale. Its `FULL_SHARD` is algorithmically ZeRO-3; `SHARD_GRAD_OP` is ZeRO-2; `NO_SHARD` is DDP; `HYBRID_SHARD` shards inside a node and replicates across nodes — a topology-aware mode ZeRO expresses differently (MiCS).
- **FSDP1 vs FSDP2.** `FullyShardedDataParallel` (FSDP1) wraps modules into `FlatParameter` blobs. **FSDP2** (`torch.distributed.fsdp.fully_shard`) is the current API: each parameter becomes a **`DTensor` sharded on dim-0**, which composes with tensor parallelism, makes per-parameter optimizer settings and `state_dict` sane, and is configured through `MixedPrecisionPolicy` / `CPUOffloadPolicy` rather than constructor flags. New code should target FSDP2; FSDP1's state-dict APIs already carry deprecation warnings pointing at `get_state_dict()` / `set_state_dict()`.
- **The auto-wrap policy** decides the sharding *granularity* — which submodules become communication units. `transformer_auto_wrap_policy` on the decoder-block class is the correct answer for LLMs, and it is not a detail: wrap too coarsely and you all-gather the whole model at once (defeating ZeRO-3); wrap too finely and you drown in tiny latency-bound collectives.
- **Prefetching** overlaps communication with computation. `BackwardPrefetch.BACKWARD_PRE` issues the next all-gather before the current backward finishes; `stage3_prefetch_bucket_size` is DeepSpeed's equivalent knob. Without it ZeRO-3 serialises compute behind comms and the throughput collapse is dramatic.
- **`reshard_after_forward`** is ZeRO-3's central time/space trade: `True` frees gathered parameters after the forward and re-gathers them in the backward (minimum memory, `1.5×` DDP comm volume); `False` keeps them (DDP comm volume, ZeRO-2 memory).

### One-sentence definition of the mechanics

> **ZeRO and FSDP eliminate the redundancy in data-parallel training by partitioning optimizer states, then gradients, then parameters across ranks — reconstructing each layer's parameters on demand with an all-gather immediately before use and discarding them immediately after — so that per-rank memory falls as `1/world_size` at the cost of roughly `1.5×` DDP's communication volume, with optional offload of the partitions to host RAM or NVMe.**

### The exact engineering problem it solves

- **The optimizer state, not the model, is what does not fit.** Mixed-precision AdamW costs **16 bytes per parameter** — `2` (bf16 params) `+ 2` (bf16 grads) `+ 12` (fp32 master + momentum + variance). A 7 B model needs ~112 GB before a single activation is allocated: **it does not fit on an 80 GB H100, and 12 of those 16 bytes are optimizer state.** Sharding attacks the largest term first, which is why ZeRO-1 alone already buys most of the win.
- **Gradient accumulation cannot help.** It reduces *activation* memory by shrinking the micro-batch; model state is per-step-invariant. Practitioners hit the ceiling, reduce batch size, hit it again, and conclude the model is impossible — when the fix was orthogonal all along.
- **The parallelism must not leak into the model code.** Both frameworks are wrappers, not rewrites: the forward stays `model(**batch)`. That is what makes it a *deployment* decision rather than a modelling one, and what lets one script serve 1 GPU and 128.
- **LoRA makes most of this irrelevant — say so out loud.** With a frozen 4-bit base and a 0.5 % trainable adapter, `K·Ψ` is already tiny, so ZeRO-1/2 shard nearly nothing. **ZeRO/FSDP are full-fine-tuning technology.** If your plan is QLoRA, the honest answer is that you probably need neither, and this notebook full-fine-tunes precisely so the comparison means something.

---

### The Human Element — datasets, and why one of these is the *wrong* benchmark

For a sharding comparison the corpus is not the subject — it is the **thing you must hold rigid**, and rigidly in a specific way: **every rank must receive exactly the same number of tokens in exactly the same number of steps.** Collective operations are synchronous; a rank that finishes its shard early sits in a barrier while the others keep issuing all-gathers, and the job **hangs rather than errors**. That failure mode is the most common first encounter people have with distributed training.

| HF path | What it is | Why this shape, for this technique |
|---|---|---|
| **`Salesforce/wikitext`**<br>config `wikitext-103-raw-v1` | ~100 M tokens of raw Wikipedia prose; the standard language-modelling benchmark corpus. | **The control, and what this notebook uses.** Raw text packs into **exactly fixed-length blocks**, so every step on every rank forwards an identical token count. That is the precondition for tokens/sec being comparable *across frameworks* — and for peak-memory numbers to be comparable at all, since activation memory is a direct function of `batch × seq`. It is also small enough to tokenise once, on rank 0, and hand to every rank as a memory-mapped Arrow file. |
| **`HuggingFaceFW/fineweb-edu`**<br>config `sample-10BT` | High-quality general web text, streamable at any scale. | **The realistic case at cluster scale**, where the corpus exceeds local disk and `IterableDataset` sharding becomes mandatory. It is also where the desync hazard is sharpest: streaming shards are not guaranteed equal length, so ranks can exhaust at different steps. This is what `split_dataset_by_node` and a hard `max_steps` exist to defend against. |
| **`databricks/databricks-dolly-15k`**<br>`train`, 15,011 rows | Human-written instruction/context/response triples — the honest SFT workload. | **Included as the counter-example.** It is *strongly bimodal* in length (short instructions, long context passages), so batch-to-batch token counts swing wildly. Benchmark two sharding strategies on it and the throughput delta you measure is substantially **batch composition**, not communication. Use it to train; do not use it to benchmark — or pack it into fixed blocks first, which is the same thing as using WikiText. |

> The takeaway generalises past this notebook: **if your benchmark corpus has length variance, your distributed benchmark has an uncontrolled variable.** Pack first, measure second.

---

## 2. Architectural Context Block

### **[Context Block]**

#### The 'Why' — the engineering and mathematical reason for this implementation

- **The redundancy argument, precisely.** In DDP every rank holds an identical copy of `(2 + 2 + K)Ψ` bytes and computes an identical optimizer update. Across `P` ranks that is `P−1` copies of pure waste. ZeRO's insight is that a rank only needs the *slice* it updates, plus whatever it is currently computing with — and "currently" is per-layer, not per-model. Everything else follows.
- **Why the all-gather is affordable: arithmetic intensity.** A transformer layer's forward+backward is `O(6·d²·B·L)` FLOPs against `O(d²)` bytes of parameter to gather — the compute-to-communication ratio grows with `batch × seq`. That is the entire reason ZeRO-3 is viable at all, and it is also the reason it **stops** being viable at small batch or short sequence: shrink `B·L` and you push the layer from compute-bound to communication-bound, at which point ZeRO-3 is slower than ZeRO-2 for no memory benefit you needed.
- **Communication volume, exactly.** For a ring all-reduce each rank moves `2(P−1)/P · bytes ≈ 2·bytes`:

  | Strategy | Per-step traffic | Relative to DDP |
  |---|---|---|
  | DDP | all-reduce grads | `1.0×` |
  | ZeRO-1 / ZeRO-2 | reduce-scatter grads + all-gather updated params | **`1.0×`** — same volume, different shape |
  | ZeRO-3 / FSDP `FULL_SHARD` | + all-gather params in forward **and** in backward | **`1.5×`** |

  So ZeRO-1 and ZeRO-2 are close to free: same bytes on the wire as DDP, strictly less memory. **ZeRO-1 is the default that most jobs should start at, and it is dramatically under-used.**
- **Why the auto-wrap policy is the most important knob.** Sharding granularity determines both peak memory and collective size. Wrapping at the **decoder-block** level means at most one block's parameters are materialised at a time — peak transient is `Ψ_block`, not `Ψ`. Wrapping the whole model in one unit gathers everything and gives you ZeRO-3's communication cost with ZeRO-1's memory profile: the worst of both.
- **Why offload is a different technique that looks like the same one.** `offload_optimizer` moves the `KΨ` term to host RAM and runs the optimizer step **on the CPU**. That works at `P = 1`, scales with host RAM rather than GPU count, and costs PCIe round-trips per step plus a CPU-bound Adam. `offload_param` (ZeRO-3 only) additionally streams parameters. **Never present an offload result as a sharding result** — this notebook keeps them in separate columns.
- **Why FSDP2's `DTensor` representation matters beyond aesthetics.** FSDP1 flattens each wrapped unit into one `FlatParameter`, which makes per-parameter optimizer hyperparameters, parameter freezing, and readable checkpoints awkward, and does not compose with tensor parallelism. FSDP2 shards each parameter individually as a `DTensor` on dim-0, so `2-D` parallelism (FSDP × TP) and clean `state_dict`s come for free. This is the reason to prefer FSDP2 in new code even though FSDP1 still works.

#### VRAM & Compute Impact

The two axes, stated separately (the notebook's central discipline):

| Axis | Formula | Scales with | Works at `P = 1`? |
|---|---|---|---|
| **Sharding** | `state_bytes / P` for the partitioned terms | `world_size` | **No — factor is 1** |
| **Offload** | partitioned terms leave VRAM entirely | host RAM / NVMe capacity | **Yes** |

Concretely, per rank, for AdamW mixed precision at `16 bytes/param`:

| Ψ | DDP | ZeRO-1 `P=8` | ZeRO-2 `P=8` | ZeRO-3 `P=8` | ZeRO-3 `P=64` |
|---|---|---|---|---|---|
| **0.135 B** (SmolLM2-135M) | 2.2 GB | 0.74 GB | 0.51 GB | 0.27 GB | 0.03 GB |
| **0.494 B** (Qwen2.5-0.5B) | 7.9 GB | 2.7 GB | 1.9 GB | 0.99 GB | 0.12 GB |
| **8.03 B** (Llama-3.1-8B) | 128 GB | 44 GB | 30 GB | 16 GB | 2.0 GB |
| **70.6 B** (Llama-3.3-70B) | 1,130 GB | 388 GB | 265 GB | 141 GB | 18 GB |

*(These are the exact values Step 1's calculator prints — it is the same arithmetic, not a separate estimate.)*

*(Model state only. Activations are on top and are governed by `batch × seq × layers`, which is what gradient checkpointing and sequence/context parallelism attack — a different axis again.)*

- **Throughput does not scale linearly, and the shape of the loss differs.** ZeRO-1/2 typically retain 90–95 % of DDP's per-GPU throughput. ZeRO-3/FSDP `FULL_SHARD` typically land at 70–85 % on good interconnect and **far worse on PCIe-only nodes**, because the `1.5×` volume now runs at ~1/10 the bandwidth of NVLink. *Interconnect is the variable that decides whether ZeRO-3 is brilliant or unusable*, and it is the one people forget to report.
- **CPU offload is a cliff, not a slope.** Expect **2–10× slower steps** — the optimizer step runs on CPU and every parameter crosses PCIe. It is a *capability* switch ("this run is now possible"), not a *performance* one. DeepSpeed's fused `DeepSpeedCPUAdam` is what makes it merely bad instead of catastrophic; note it **JIT-compiles on first use** (minutes), or set `zero_force_ds_cpu_optimizer: false` to fall back to torch AdamW on CPU.
- **Activation checkpointing composes with both** and is usually mandatory above ~7 B: it is orthogonal, attacks the other term, and both frameworks expose it (`fsdp_activation_checkpointing`, DeepSpeed's `activation_checkpointing` block).

#### Pros & Cons

**DeepSpeed ZeRO — pros**
- **The most complete memory toolkit that exists**: stages 1/2/3, CPU **and NVMe** offload (ZeRO-Infinity), MiCS, ZeRO++ quantised collectives, fused CPU Adam, its own fp16 loss scaler.
- **JSON configuration** — the run is a version-controlled artifact, and `"auto"` values let HF's `Trainer` fill in batch/LR/scheduler consistently.
- **Stage 3 + NVMe offload is the only mainstream path to training models far larger than aggregate GPU memory.**
- Mature multi-node tooling (`hostfile`, its own launcher, elastic training).

**DeepSpeed ZeRO — cons**
- **A large third-party dependency that owns your training loop.** `deepspeed.initialize` returns an engine; `engine.backward(loss)` and `engine.step()` replace the PyTorch idioms, and the LR scheduler and grad-clipping move inside it.
- **Version coupling to torch/CUDA is real**, and JIT-compiled ops (`cpu_adam`, fused kernels) surprise people on first run and in containers.
- **The config surface is enormous** and cross-key interactions are under-documented; several keys are silently deprecated aliases (`stage3_gather_fp16_weights_on_model_save` → `stage3_gather_16bit_weights_on_model_save`).
- **Checkpoints are sharded and framework-specific.** Getting a plain `pytorch_model.bin` needs `stage3_gather_16bit_weights_on_model_save: true` or a post-hoc `zero_to_fp32.py` run.

**PyTorch FSDP — pros**
- **Native.** No extra dependency, ships and versions with torch, and it is where PyTorch's distributed investment goes.
- **FSDP2 composes**: `DTensor` sharding stacks with tensor/pipeline/context parallelism into real n-D parallelism. DeepSpeed's story here is weaker.
- **Ordinary PyTorch semantics** — `loss.backward()`, your own optimizer, your own scheduler. Debugging is debugging PyTorch.
- **Distributed Checkpointing (DCP)** saves/loads sharded state and can **reshard across a different `world_size`**, which matters more than it sounds when your cluster allocation changes.
- Fine-grained, *typed* control: `MixedPrecisionPolicy`, `CPUOffloadPolicy`, `reshard_after_forward`, per-module wrapping.

**PyTorch FSDP — cons**
- **No NVMe offload**, and CPU offload is coarser than DeepSpeed's. Beyond aggregate GPU + host RAM, FSDP has no answer.
- **You must choose the wrap policy correctly.** DeepSpeed infers granularity; FSDP hands you the footgun. A wrong `transformer_layer_cls_to_wrap` silently produces bad performance, not an error.
- **Two live APIs.** FSDP1 and FSDP2 coexist with different configuration surfaces, and most blog posts, Stack Overflow answers and older `accelerate` configs are FSDP1.
- **Checkpointing is more ceremony**: `get_state_dict`/`set_state_dict` or DCP, and `FULL_STATE_DICT` on a large model can OOM rank 0.

**Choosing (the actual architectural decision):**
- **Need NVMe offload, or the absolute largest model per dollar** → DeepSpeed ZeRO-3.
- **Composing with tensor/pipeline parallel, or you want to stay in-framework** → FSDP2.
- **Model fits with ZeRO-1/2 memory** → use ZeRO-1 or `SHARD_GRAD_OP`; do not pay ZeRO-3's `1.5×` for capacity you did not need.
- **Using LoRA/QLoRA** → you probably need neither; the optimizer state is already small.
- **Undecided** → FSDP2, because "no extra dependency" is worth real money in maintenance, and `accelerate` makes the switch a config change either way.

#### Metrics to watch

- **Peak VRAM per rank — `max_memory_allocated` *and* `max_memory_reserved`, reduced with `MAX` across ranks.** The slowest/fattest rank is what OOMs the job; a rank-0-only number is not a result.
- **Tokens/sec per GPU *and* aggregate.** Aggregate hides per-GPU regressions; per-GPU hides whether you actually gained anything by adding hardware.
- **Scaling efficiency** = `throughput(P) / (P × throughput(1))`. Below ~0.7 you are communication-bound and should reconsider the stage before buying GPUs.
- **The analytical comm volume** (`1.0×` vs `1.5×` DDP) next to the measured throughput. Together they tell you whether a slowdown is expected physics or a misconfiguration.
- **Host RAM high-water mark whenever offload is on** — the second, invisible OOM, and pinned memory makes it worse.
- **Time-to-first-step.** ZeRO-3 and FSDP have real initialisation costs (sharding, `zero.Init`, meta-device materialisation) that never show up in steady-state throughput but dominate short jobs.
- **Loss parity against the single-GPU run.** Sharding must be numerically neutral. A different loss curve means a wrong `reduce_dtype`, a broken gradient-accumulation boundary, or a desynced sampler — not a "distributed training effect."

---

## 3. Production-Grade Implementation

**Full fine-tune of a small Llama/Qwen-class model on packed WikiText-103 blocks, run through DDP, DeepSpeed ZeRO-1/2/3 and PyTorch FSDP2 — the same training loop, the same data, the same optimizer, only the wrapping changes.** The model is chosen by the probe (`SmolLM2-135M` on a 16 GB card, `Qwen2.5-0.5B` on ≥24 GB), for a reason Step 2 makes explicit and that is worth more than the benchmark itself.

> ⚠️ **What this notebook can and cannot prove on your hardware.** The probe in the next cell prints a verdict; here is the rule it applies:
>
> | Your box | Sharding axis | Offload axis | What you get |
> |---|---|---|---|
> | **1 GPU** (Colab T4) | not measurable — the factor is literally `1` | **fully measurable** | a real offload result, a wiring check for every backend, and exact analytical sharding numbers |
> | **≥ 2 GPUs** | **measurable** | measurable | the complete head-to-head matrix |
>
> The notebook does **not** fake the single-GPU case by reporting a "saving" that is really offload. That conflation is the single most common error in this topic, and refusing to make it is the point.
>
> ⚙️ **Why a raw training loop instead of `Trainer`.** `transformers.Trainer` integrates both backends and is what you would ship — but it also inserts its own accumulation, scaling and checkpointing logic *between* you and the thing being compared. A ~50-line loop that is byte-identical across all three backends makes the comparison exact. The `Trainer`/`accelerate` equivalents are generated in Step 5 as the artifacts you would actually deploy.

**Executable pipeline:**

| Step | What | The point |
|---|---|---|
| 0 | Install + **topology probe** | GPUs, per-GPU VRAM, host RAM, interconnect — and a verdict on what is measurable here |
| 1 | **Analytical memory model** | the sharding axis, exactly, for any Ψ and any `P`; no cluster needed |
| 2 | WikiText-103 → fixed-length packed blocks, tokenised **once** | equal tokens per rank per step, or the job hangs |
| 3 | `ds_config_stage{1,2,3}.json` | the DeepSpeed artifact, every key annotated |
| 4 | `train_shard.py` — one loop, three backends, FSDP2 wrapping in Python | the comparison, and the `fully_shard` code the chapter asks for |
| 5 | `torchrun` + `accelerate` launch artifacts, single- and multi-node | how it actually ships |
| 6 | Run the matrix, table + plot | measured VRAM and throughput |
| 7 | **Checkpointing** — the part that breaks in production | sharded → consolidated, both frameworks |

### Environment Setup

In [ ]:
# DeepSpeed ships prebuilt wheels for common torch/CUDA pairs; ZeRO's CPU-offload optimizer
# (`cpu_adam`) is JIT-compiled on FIRST USE and takes minutes — Step 3 shows the escape hatch.
%pip install -q deepspeed
%pip install -q --upgrade "transformers>=4.45" "accelerate>=1.0" datasets psutil matplotlib

In [ ]:
import json
import os
import subprocess
from pathlib import Path

import psutil
import torch

# ------------------------------------------------------------------ topology probe
N_GPU = torch.cuda.device_count()
CC = torch.cuda.get_device_capability(0) if N_GPU else (0, 0)
SM = CC[0] * 10 + CC[1]
GPU_NAME = torch.cuda.get_device_name(0) if N_GPU else "CPU"
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 1e9 if N_GPU else 0.0
HOST_GB = psutil.virtual_memory().total / 1e9

# bf16 needs Ampere. On Turing we run pure fp32 for BOTH backends rather than fp16: FSDP2 would
# need a ShardedGradScaler while DeepSpeed brings its own loss scaler, and that asymmetry would
# quietly make the two arms numerically different runs. fp32 keeps the comparison honest.
SUPPORTS_BF16 = SM >= 80
DTYPE = "bfloat16" if SUPPORTS_BF16 else "float32"

# Pinned host memory is much faster for offload but is non-swappable; on a small box it is the
# fastest route to an invisible host-RAM OOM.
PIN_MEMORY = HOST_GB >= 24

print(f"gpus        : {N_GPU} x {GPU_NAME} ({GPU_GB:.1f} GB each, sm_{SM})")
print(f"host RAM    : {HOST_GB:.1f} GB")
print(f"dtype       : {DTYPE}   (bf16 needs sm_80+)")
print(f"offload pin : {PIN_MEMORY}")

if N_GPU >= 2:
    print(subprocess.run(["nvidia-smi", "topo", "-m"], capture_output=True, text=True).stdout[:900])

print("\n" + "=" * 78)
if N_GPU >= 2:
    print(f"VERDICT: {N_GPU} GPUs — the full matrix is measurable.")
    print("  Sharding axis: REAL (factor = world_size).   Offload axis: REAL.")
elif N_GPU == 1:
    print("VERDICT: 1 GPU — sharding is NOT measurable here and this notebook will not pretend.")
    print("  Sharding axis: factor = 1, i.e. ZeRO-1/2/3 save exactly nothing. Step 1 gives the")
    print("                 exact numbers analytically instead of guessing at them.")
    print("  Offload axis : REAL and measured — CPU offload works at world_size=1.")
    print("  Also verified: every backend's wiring, wrap policy and checkpoint path.")
else:
    print("VERDICT: no GPU — only Step 1 (the analytical model) will run.")
print("=" * 78)

WORKDIR = Path("/content") if Path("/content").exists() else Path.cwd()
os.chdir(WORKDIR)

### Step 1 — The analytical memory model

This is arithmetic, not benchmarking, so it is exactly right on any machine — including one with no GPU at all. **Run it before you book a cluster**, because it answers the only question that matters at planning time: *what is the smallest configuration on which this model trains?*

The accounting, per parameter, for mixed-precision AdamW:

| Term | Bytes | Sharded by |
|---|---|---|
| Parameters (compute dtype) | 2 | ZeRO-3 / `FULL_SHARD` only |
| Gradients (compute dtype) | 2 | ZeRO-2 and up |
| fp32 master copy | 4 | ZeRO-1 and up |
| Adam momentum `m` | 4 | ZeRO-1 and up |
| Adam variance `v` | 4 | ZeRO-1 and up |
| **Total** | **16** | |

Two things the table below makes obvious and that most write-ups bury:

1. **ZeRO-1 does most of the work.** `12` of the `16` bytes are optimizer state, so stage 1 alone removes ~75 % of the redundant memory — at **DDP's exact communication volume**. Going straight to stage 3 is a common and expensive reflex.
2. **In pure fp32 (no mixed precision) the total is still 16 bytes** — `4` param `+ 4` grad `+ 4 m + 4 v`, with no master copy needed. Convenient for this notebook, since a T4 runs fp32.

In [ ]:
BYTES = {"param": 2 if SUPPORTS_BF16 else 4, "grad": 2 if SUPPORTS_BF16 else 4,
         "master": 4 if SUPPORTS_BF16 else 0, "m": 4, "v": 4}
K_OPT = BYTES["master"] + BYTES["m"] + BYTES["v"]      # the 12 (or 8) that ZeRO-1 shards

def per_rank_gb(n_params, world_size, stage, offload=""):
    """Model-state bytes resident on ONE GPU. Activations are on top of this."""
    p, g, o = BYTES["param"], BYTES["grad"], K_OPT
    if stage >= 1: o /= world_size   # ZeRO-1: optimizer states
    if stage >= 2: g /= world_size   # ZeRO-2: + gradients
    if stage >= 3: p /= world_size   # ZeRO-3: + parameters
    if "optimizer" in offload: o = 0.0  # offload leaves VRAM entirely (-> host RAM)
    if "param" in offload:     p = 0.0
    return n_params * (p + g + o) / 1e9

MODELS = [("SmolLM2-135M", 0.135e9), ("Qwen2.5-0.5B", 0.494e9),
          ("Llama-3.1-8B", 8.03e9), ("Llama-3.3-70B", 70.6e9)]
STAGES = [(0, "DDP (no sharding)"), (1, "ZeRO-1  optim"), (2, "ZeRO-2  +grad"), (3, "ZeRO-3  +param")]

for label, n in MODELS:
    print(f"\n{label}  ({n/1e9:.2f}B params, {BYTES['param']+BYTES['grad']+K_OPT} bytes/param)")
    print(f"  {'strategy':<22}" + "".join(f"{f'P={p}':>11}" for p in (1, 2, 8, 64)))
    print("  " + "-" * 66)
    for stage, name in STAGES:
        row = "".join(f"{per_rank_gb(n, p, stage):>11.2f}" for p in (1, 2, 8, 64))
        print(f"  {name:<22}{row}")
    print(f"  {'ZeRO-3 + CPU offload':<22}"
          + "".join(f"{per_rank_gb(n, p, 3, 'optimizer+param'):>11.2f}" for p in (1, 2, 8, 64))
          + "   <- host RAM pays instead")

print("\nGB of model state per GPU. Read the P=1 column: every ZeRO stage is IDENTICAL there.")
print("That column is the whole reason a single-GPU box cannot measure the sharding axis.")

if N_GPU:
    n = 0.494e9
    print(f"\nThis box: {GPU_GB:.1f} GB/GPU x {N_GPU}. Qwen2.5-0.5B full fine-tune needs "
          f"{per_rank_gb(n, N_GPU, 0):.2f} GB of model state per GPU under DDP "
          f"({per_rank_gb(n, N_GPU, 3, 'optimizer+param'):.2f} GB with ZeRO-3+offload), before activations.")

### Step 2 — The corpus: pack once, on one process

Two properties are non-negotiable for a distributed benchmark, and both are about **synchrony**:

- **Fixed-length blocks.** Every step forwards `batch × 1024` tokens on every rank, so tokens/sec compares across backends and activation memory is constant.
- **Tokenise once, memory-map everywhere.** The Arrow file is written here, in the notebook, and each rank opens it read-only. Ranks that each tokenise the corpus independently waste `world_size − 1` copies of the CPU work and can race on the cache lock — the distributed cold-start bug.

The third property is enforced in the training script rather than here: `DistributedSampler(..., drop_last=True)`. **Without `drop_last`, ranks receive different step counts and the job hangs in a collective** — not an error, a hang. It is the first thing to check when a distributed run stops producing logs.

> ⚠️ **Size the batch shape before you size the sharding — the loss term is not the model term.** Cross-entropy materialises three full-size fp32 tensors: the `(bs·seq, vocab)` logits, their log-softmax, and their gradient. This is **independent of parameter count** and linear in `bs`, `seq` *and* `vocab`:
>
> | Model | vocab | state (16 B/param) | loss term @ `bs=2, seq=1024` | total |
> |---|---|---|---|---|
> | Qwen2.5-0.5B | 151,936 | 7.9 GB | **3.7 GB** | 11.6 GB → **OOMs a 14.5 GB T4** |
> | SmolLM2-135M | 49,152 | 2.2 GB | 1.2 GB | 3.4 GB → comfortable |
>
> Qwen2.5-0.5B is the smaller-*sounding* choice and the one that does not fit, because its vocabulary is 3× larger. The cell below prints this budget **before** anything is launched and warns if it exceeds 75 % of the card. A sharding benchmark whose baseline cannot run measures nothing.

In [ ]:
from datasets import Dataset, load_dataset
from transformers import AutoConfig, AutoTokenizer

SEQ_LEN = 1024
MICRO_BS = 2
N_BLOCKS = 512    # enough for ~60 steps at batch 2; raise on real hardware

# ---- Model choice is a function of the CARD, and the deciding term is the LOSS ----------
# Cross-entropy materialises (bs*seq, vocab) logits, their log-softmax, AND their gradient --
# three full-size fp32 tensors. At vocab 151,936 with bs=2 x seq=1024 that is ~3.7 GB stacked
# on top of 7.9 GB of model state, which is precisely how a "small" 0.5B model OOMs a 14.5 GB
# T4 before sharding has been measured at all. Size the batch shape FIRST; the sharding
# experiment is meaningless if the baseline cannot run.
if GPU_GB >= 24 and HOST_GB >= 32:
    MODEL_ID, LAYER_CLS, N_PARAMS = "Qwen/Qwen2.5-0.5B", "Qwen2DecoderLayer", 0.494e9
else:
    MODEL_ID, LAYER_CLS, N_PARAMS = "HuggingFaceTB/SmolLM2-135M", "LlamaDecoderLayer", 0.135e9

tok = AutoTokenizer.from_pretrained(MODEL_ID)
VOCAB = AutoConfig.from_pretrained(MODEL_ID).vocab_size

state_gb = N_PARAMS * 16 / 1e9   # 4+4+4+4 bytes/param in fp32
loss_gb = 3 * MICRO_BS * SEQ_LEN * VOCAB * 4 / 1e9   # logits + log_softmax + grad
print(f"model: {MODEL_ID}  ({N_PARAMS/1e6:.0f}M params, vocab {VOCAB:,})")
print(f"wrap class: {LAYER_CLS}")
print(f"model state: {state_gb:.2f} GB   (16 bytes/param, AdamW fp32)")
print(f"loss term: {loss_gb:.2f} GB   (bs={MICRO_BS} x seq={SEQ_LEN} x vocab, THREE fp32 copies)")
print(f"budget: {state_gb + loss_gb:.2f} GB of {GPU_GB:.1f} GB card, before activations")
if state_gb + loss_gb > 0.75 * GPU_GB:
    print("  !! over 75% of the card before activations — lower MICRO_BS or SEQ_LEN")
print()
wiki = load_dataset("Salesforce/wikitext", "wikitext-103-raw-v1", split="train")
print(wiki)

# Concat-and-chunk: the pretraining-standard packer. ~100% token utilisation and, more
# importantly here, EXACTLY SEQ_LEN tokens per row so every rank's every step is identical.
ids, i = [], 0
for text in wiki["text"]:
    if len(text) < 200:   # skip section headers and blank lines
        continue
    ids.extend(tok(text, add_special_tokens=False)["input_ids"] + [tok.eos_token_id])
    i += 1
    if len(ids) >= SEQ_LEN * N_BLOCKS:
        break

blocks = [ids[i * SEQ_LEN:(i + 1) * SEQ_LEN] for i in range(len(ids) // SEQ_LEN)][:N_BLOCKS]
ds = Dataset.from_dict({"input_ids": blocks, "labels": [b[:] for b in blocks]})
ds.set_format("torch")
DATA_DIR = str(WORKDIR / "wikitext_blocks")
ds.save_to_disk(DATA_DIR)

print(f"\ndocuments consumed: {i:,}")
print(f"blocks: {len(blocks)} x {SEQ_LEN} tokens = {len(blocks)*SEQ_LEN/1e6:.2f}M tokens")
print(f"saved to: {DATA_DIR}  (memory-mapped by every rank)")

### Step 3 — The DeepSpeed artifact: `ds_config_stage{1,2,3}.json`

DeepSpeed is configured declaratively, and the config **is** the experiment — commit it. Every key below is annotated because in practice each one is either load-bearing or a known trap.

Two that deserve attention before you run anything:

- **`zero_force_ds_cpu_optimizer: false`.** With `offload_optimizer` enabled, DeepSpeed insists on its fused `DeepSpeedCPUAdam`, which **JIT-compiles on first use** — several minutes, and it needs a matching CUDA toolchain. Setting this to `false` falls back to torch's CPU AdamW: slower per step, but it starts *now* and it works in slim containers. Flip it to `true` for a real run.
- **`"auto"` values.** Inside `transformers.Trainer`, keys like `train_micro_batch_size_per_gpu` and the scheduler block accept `"auto"` and are filled from `TrainingArguments`, which prevents the classic silent contradiction between the two configs. This notebook's raw loop sets them explicitly, since there is no `Trainer` to infer from.

In [ ]:
def ds_config(stage, offload_optimizer=False, offload_param=False,
              micro_bs=2, accum=1, world=1, lr=2e-5):
    """Emit a ZeRO config. Every non-obvious key is annotated in the comments below."""
    zero = {
        "stage": stage,

        # ---- Communication shaping (stages 1-3) ------------------------------------------
        # Overlap reduce-scatter/all-gather with backward compute. Off => comms serialise
        # behind compute and ZeRO-3 throughput falls off a cliff.
        "overlap_comm": True,
        # Copy gradients into one contiguous buffer as they are produced: avoids the memory
        # fragmentation that makes long ZeRO-2/3 runs OOM at step 900 rather than step 1.
        "contiguous_gradients": True,
        # Collective bucket sizes. Bigger = fewer, larger messages (better on NVLink);
        # smaller = more overlap opportunity (better on slow PCIe).
        "reduce_bucket_size": 5e8,
        "allgather_bucket_size": 5e8,
    }

    if stage == 3:
        # ---- Stage-3-only: when to fetch, when to keep, when to free --------------------
        zero.update({
            # Prefetch the NEXT layer's parameters while the current one computes. Without
            # this, ZeRO-3 serialises every all-gather behind compute.
            "stage3_prefetch_bucket_size": 5e7,
            # Parameters smaller than this stay resident instead of being re-gathered every
            # step -- biases and norms are tiny and re-gathering them is pure latency.
            "stage3_param_persistence_threshold": 1e5,
            # Cap on parameters materialised at once: the real peak-memory knob for stage 3.
            "stage3_max_live_parameters": 1e9,
            "stage3_max_reuse_distance": 1e9,
            # Consolidate the sharded params into a normal fp16/bf16 checkpoint on save.
            # WITHOUT this you get shards and must post-process with zero_to_fp32.py (Step 7).
            "stage3_gather_16bit_weights_on_model_save": True,
        })

    if offload_optimizer:
        # The KΨ term leaves VRAM for host RAM; the optimizer STEP now runs on the CPU.
        zero["offload_optimizer"] = {"device": "cpu", "pin_memory": PIN_MEMORY}
    if offload_param:
        # Stage 3 only. Parameters stream host->device per layer. Deepest memory saving,
        # steepest throughput cost. `"device": "nvme"` + `nvme_path` is ZeRO-Infinity.
        zero["offload_param"] = {"device": "cpu", "pin_memory": PIN_MEMORY}

    cfg = {
        "train_micro_batch_size_per_gpu": micro_bs,
        "gradient_accumulation_steps": accum,
        # DeepSpeed derives train_batch_size = micro * accum * world. Setting all three
        # inconsistently is a hard error at init -- one of its better design choices.
        "train_batch_size": micro_bs * accum * world,

        "gradient_clipping": 1.0,
        "steps_per_print": 10**9,   # the raw loop does its own logging
        "wall_clock_breakdown": False,   # True prints a per-phase comms/compute breakdown

        "zero_optimization": zero,
        # Use torch's CPU AdamW instead of JIT-compiling DeepSpeedCPUAdam (minutes on first
        # run). Set True for production: the fused kernel is several times faster.
        "zero_force_ds_cpu_optimizer": False,

        "optimizer": {"type": "AdamW",
                      "params": {"lr": lr, "betas": [0.9, 0.999], "eps": 1e-8, "weight_decay": 0.01}},
    }
    # DeepSpeed owns loss scaling in fp16; in bf16 no scaler is needed; in fp32 neither block
    # is present. This is a genuine DeepSpeed convenience over raw FSDP.
    if SUPPORTS_BF16:
        cfg["bf16"] = {"enabled": True}
    return cfg


CONFIGS = {}
for stage in (1, 2, 3):
    p = WORKDIR / f"ds_config_stage{stage}.json"
    p.write_text(json.dumps(ds_config(stage), indent=2))
    CONFIGS[f"stage{stage}"] = p

p = WORKDIR / "ds_config_stage3_offload.json"
p.write_text(json.dumps(ds_config(3, offload_optimizer=True, offload_param=True), indent=2))
CONFIGS["stage3_offload"] = p

p = WORKDIR / "ds_config_stage2_offload.json"
p.write_text(json.dumps(ds_config(2, offload_optimizer=True), indent=2))
CONFIGS["stage2_offload"] = p

print("\n".join(f"{k:<18} {v.name}" for k, v in CONFIGS.items()))
print("\n--- ds_config_stage3_offload.json ---")
print((WORKDIR / "ds_config_stage3_offload.json").read_text())

### Step 4 — `train_shard.py`: one loop, three backends

The script below is the comparison. The data loading, the model, the optimizer hyperparameters, the step count and the timing instrumentation are **byte-identical across all three backends**; only the wrapping differs. That is the only way a throughput number means anything.

The FSDP section is the code the chapter asks for, written out rather than delegated to a config:

```python
mp = MixedPrecisionPolicy(param_dtype=torch.bfloat16, reduce_dtype=torch.float32)
for layer in model.model.layers:                  # <- the auto-wrap policy, made explicit
    fully_shard(layer, mp_policy=mp, offload_policy=off, reshard_after_forward=True)
fully_shard(model, mp_policy=mp, offload_policy=off)   # root last
```

Four details in there that are the whole game:

- **Wrapping per decoder block** is the transformer auto-wrap policy stated directly. Peak transient parameter memory becomes one block's worth, not the model's. `accelerate`'s `fsdp_transformer_layer_cls_to_wrap: Qwen2DecoderLayer` compiles to exactly this loop.
- **The root module is wrapped last**, after its children — FSDP2 builds the parameter groups bottom-up, and wrapping the root first swallows everything into one shard unit.
- **`reshard_after_forward=True`** frees gathered parameters after the forward and re-gathers in the backward: minimum memory, `1.5×` DDP comm. Set `False` for ZeRO-2-like behaviour at DDP comm volume.
- **`reduce_dtype=torch.float32`** keeps gradient reduction in fp32 even when parameters are bf16. Reducing in bf16 across many ranks accumulates rounding error and is a real source of "distributed training diverges but single-GPU doesn't."

In [ ]:
train_py = r"""
import argparse, json, os, time

import torch
import torch.distributed as dist
from datasets import load_from_disk
from torch.utils.data import DataLoader, DistributedSampler
from transformers import AutoModelForCausalLM

ap = argparse.ArgumentParser()
ap.add_argument("--backend", choices=["ddp", "deepspeed", "fsdp"], required=True)
ap.add_argument("--ds-config", default=None, help="path to ds_config.json (deepspeed only)")
ap.add_argument("--fsdp-offload", action="store_true", help="CPUOffloadPolicy (fsdp only)")
ap.add_argument("--pin-memory", default="false", help="pin offloaded host memory")
ap.add_argument("--reshard-after-forward", default="true")
ap.add_argument("--model", default="HuggingFaceTB/SmolLM2-135M")
ap.add_argument("--data", required=True)
ap.add_argument("--steps", type=int, default=12)
ap.add_argument("--warmup", type=int, default=3)
ap.add_argument("--bs", type=int, default=2)
ap.add_argument("--seq", type=int, default=1024)
ap.add_argument("--lr", type=float, default=2e-5)
ap.add_argument("--tag", default="run")
ap.add_argument("--out", default="shard_results.jsonl")
args = ap.parse_args()

LOCAL_RANK = int(os.environ.get("LOCAL_RANK", 0))
RANK = int(os.environ.get("RANK", 0))
WORLD = int(os.environ.get("WORLD_SIZE", 1))
torch.cuda.set_device(LOCAL_RANK)
DEV = torch.device("cuda", LOCAL_RANK)
BF16 = torch.cuda.get_device_capability(LOCAL_RANK)[0] >= 8
DTYPE = torch.bfloat16 if BF16 else torch.float32
torch.manual_seed(0)

if args.backend == "deepspeed":
    import deepspeed
    deepspeed.init_distributed(dist_backend="nccl")
else:
    dist.init_process_group("nccl")

# ---- DATA: identical bytes on every rank, memory-mapped, EQUAL STEPS PER RANK -----------
ds = load_from_disk(args.data)
ds.set_format("torch", columns=["input_ids", "labels"])
sampler = DistributedSampler(ds, num_replicas=WORLD, rank=RANK, shuffle=True,
                             # drop_last=True is NOT an optimisation. Without it ranks get
                             # different step counts and the job HANGS inside a collective.
                             drop_last=True, seed=0)
loader = DataLoader(ds, batch_size=args.bs, sampler=sampler, num_workers=2, pin_memory=True,
                    drop_last=True)

# ---- MODEL --------------------------------------------------------------------------
# NOTE for production: at 7B+ every rank materialising the full model before sharding is
# itself an OOM. The fixes are `deepspeed.zero.Init()` (ZeRO-3) or meta-device init plus
# `sync_module_states=True` (FSDP). At 0.5B the naive path is fine and clearer.
model = AutoModelForCausalLM.from_pretrained(args.model, dtype=torch.float32)
model.config.use_cache = False
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})

comm_factor = 1.0          # analytical per-step traffic relative to DDP
if args.backend == "ddp":
    from torch.nn.parallel import DistributedDataParallel as DDP
    model = model.to(DEV)
    engine = DDP(model, device_ids=[LOCAL_RANK])
    opt = torch.optim.AdamW(engine.parameters(), lr=args.lr, weight_decay=0.01)

elif args.backend == "deepspeed":
    cfg = json.loads(open(args.ds_config).read())
    cfg["train_micro_batch_size_per_gpu"] = args.bs
    cfg["train_batch_size"] = args.bs * WORLD
    # deepspeed.initialize returns an ENGINE: it owns backward(), step(), clipping and the
    # LR schedule. That ownership is the main ergonomic difference from FSDP.
    engine, opt, _, _ = deepspeed.initialize(
        model=model, model_parameters=model.parameters(), config=cfg
    )
    comm_factor = 1.5 if cfg["zero_optimization"]["stage"] == 3 else 1.0

else:  # ---- FSDP2: fully_shard, DTensor-based -----------------------------------------
    try:
        from torch.distributed.fsdp import (CPUOffloadPolicy, MixedPrecisionPolicy,
                                            OffloadPolicy, fully_shard)
    except ImportError:  # torch < 2.6 kept it under _composable
        from torch.distributed._composable.fsdp import (CPUOffloadPolicy, MixedPrecisionPolicy,
                                                        OffloadPolicy, fully_shard)

    mp = MixedPrecisionPolicy(
        param_dtype=torch.bfloat16 if BF16 else torch.float32,
        # Reduce gradients in fp32 even with bf16 params. Reducing in bf16 across many ranks
        # accumulates rounding error -- a real cause of "diverges distributed, fine on 1 GPU".
        reduce_dtype=torch.float32,
    )
    # Pinned host memory is faster but NON-SWAPPABLE: on a small-RAM box pinning the
    # offloaded state is the fastest route to an invisible SIGKILL from the OOM killer.
    # This mirrors the notebook's PIN_MEMORY probe rather than hardcoding True.
    pin = args.pin_memory.lower() == "true"
    off = CPUOffloadPolicy(pin_memory=pin) if args.fsdp_offload else OffloadPolicy()
    raf = args.reshard_after_forward.lower() == "true"

    model = model.to(DEV)
    # THE AUTO-WRAP POLICY, WRITTEN OUT: one shard unit per decoder block, so peak transient
    # parameter memory is one block -- not the whole model.
    for layer in model.model.layers:
        fully_shard(layer, mp_policy=mp, offload_policy=off, reshard_after_forward=raf)
    # Root LAST: FSDP2 groups parameters bottom-up, and wrapping the root first would put the
    # entire model into a single shard unit (ZeRO-3 comms, ZeRO-1 memory: worst of both).
    fully_shard(model, mp_policy=mp, offload_policy=off, reshard_after_forward=raf)
    engine = model
    opt = torch.optim.AdamW(engine.parameters(), lr=args.lr, weight_decay=0.01)
    comm_factor = 1.5 if raf else 1.0

# ---- TRAIN --------------------------------------------------------------------------
sampler.set_epoch(0)
it = iter(loader)
times, losses = [], []
t_start = time.perf_counter()

for step in range(args.steps + args.warmup):
    try:
        batch = next(it)
    except StopIteration:
        it = iter(loader); batch = next(it)
    batch = {k: v.to(DEV, non_blocking=True) for k, v in batch.items()}

    if step == args.warmup:
        # Measure STEADY STATE: exclude allocator warmup, autotuning and the first all-gather.
        dist.barrier(); torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
        t0 = time.perf_counter()

    out = engine(**batch)
    loss = out.loss
    if args.backend == "deepspeed":
        engine.backward(loss)      # engine owns scaling + gradient partitioning
        engine.step()              # ... and clipping + zero_grad
    else:
        loss.backward()
        torch.nn.utils.clip_grad_norm_(engine.parameters(), 1.0)
        opt.step(); opt.zero_grad(set_to_none=True)

    if step >= args.warmup:
        torch.cuda.synchronize()
        now = time.perf_counter()
        times.append(now - t0); t0 = now
        losses.append(loss.item())

# ---- REDUCE ACROSS RANKS: the SLOWEST/FATTEST rank is what OOMs and stalls the job -----
peak_alloc = torch.tensor([torch.cuda.max_memory_allocated() / 1e9], device=DEV)
peak_resv = torch.tensor([torch.cuda.max_memory_reserved() / 1e9], device=DEV)
dist.all_reduce(peak_alloc, op=dist.ReduceOp.MAX)
dist.all_reduce(peak_resv, op=dist.ReduceOp.MAX)

if RANK == 0:
    times.sort()
    median = times[len(times) // 2]
    rec = {
        "tag": args.tag, "backend": args.backend, "world_size": WORLD,
        "median_step_s": median,
        "tokens_per_sec_per_gpu": args.bs * args.seq / median,
        "tokens_per_sec_total": args.bs * args.seq * WORLD / median,
        "peak_alloc_gb": peak_alloc.item(), "peak_reserved_gb": peak_resv.item(),
        "host_ram_peak_gb": __import__("psutil").Process().memory_info().rss / 1e9,
        "comm_factor_vs_ddp": comm_factor,
        "first_loss": losses[0], "last_loss": losses[-1],
        "startup_s": t_start and (time.perf_counter() - t_start),
    }
    with open(args.out, "a") as fh:
        fh.write(json.dumps(rec) + "\n")
    print("SHARD_RESULT " + json.dumps(rec))

dist.barrier()
dist.destroy_process_group()
"""

Path("train_shard.py").write_text(train_py, encoding="utf-8")
print(f"wrote train_shard.py ({len(train_py.splitlines())} lines)")

### Step 5 — Launching: `torchrun` and `accelerate`

**`torchrun`** sets `RANK` / `LOCAL_RANK` / `WORLD_SIZE` / `MASTER_ADDR` / `MASTER_PORT` and spawns one process per GPU. Single node:

```bash
torchrun --standalone --nproc_per_node=8 train_shard.py --backend fsdp --data ./wikitext_blocks
```

Two nodes × 8 GPUs — same command on both machines, one variable apart:

```bash
torchrun --nnodes=2 --nproc_per_node=8 --node_rank=$NODE_RANK \
         --rdzv_backend=c10d --rdzv_endpoint=$MASTER_ADDR:29500 \
         train_shard.py --backend deepspeed --ds-config ds_config_stage3.json --data /shared/wikitext_blocks
```

**`accelerate`** is the layer you would actually ship: the same YAML drives `Trainer`, TRL and Axolotl, and switching DeepSpeed ↔ FSDP becomes a config change rather than a code change. The cell below writes both configs with the **verified** key names (`accelerate`'s launcher maps `fsdp_config.fsdp_*` keys onto `FSDP_*` environment variables, so the spelling is not negotiable).

Three multi-node facts that bite, and are not in either project's quickstart:

1. **The effective batch is `micro_bs × accum × world_size`.** Going 1 → 16 GPUs at fixed accumulation multiplies your batch by 16 and silently invalidates the LR you tuned. DeepSpeed at least *errors* if `train_batch_size` contradicts the other two.
2. **`NCCL_SOCKET_IFNAME` and `NCCL_IB_DISABLE`** decide whether your job runs at InfiniBand speed, at Ethernet speed, or hangs at rank 0 with no output. If a multi-node job stalls before step 1, this is the first suspect.
3. **Run the tokenisation step once, on shared storage, before the job.** Every rank re-tokenising is `world_size − 1` wasted CPU-hours and a cache-lock race.

In [ ]:
import textwrap

# ---- accelerate + DeepSpeed ------------------------------------------------------------
(WORKDIR / "acc_deepspeed.yaml").write_text(textwrap.dedent(f"""
    compute_environment: LOCAL_MACHINE
    distributed_type: DEEPSPEED
    num_machines: 1
    num_processes: {max(N_GPU, 1)}
    machine_rank: 0
    mixed_precision: {"bf16" if SUPPORTS_BF16 else "no"}
    use_cpu: false
    deepspeed_config:
      # Point at the JSON for full control; the inline keys below are the shorthand.
      deepspeed_config_file: ds_config_stage3.json
      zero3_init_flag: true          # deepspeed.zero.Init(): shard AS the model is built,
                                     # so no rank ever materialises the full model. Mandatory at 7B+.
      zero3_save_16bit_model: true   # consolidate shards into a normal checkpoint on save
    """).lstrip(), encoding="utf-8")

# ---- accelerate + FSDP2 ----------------------------------------------------------------
# Key names verified against accelerate/utils/launch.py, which maps each `fsdp_*` key onto an
# FSDP_* env var read by FullyShardedDataParallelPlugin. Spelling is not negotiable.
(WORKDIR / "acc_fsdp.yaml").write_text(textwrap.dedent(f"""
    compute_environment: LOCAL_MACHINE
    distributed_type: FSDP
    num_machines: 1
    num_processes: {max(N_GPU, 1)}
    machine_rank: 0
    mixed_precision: {"bf16" if SUPPORTS_BF16 else "no"}
    use_cpu: false
    fsdp_config:
      fsdp_version: 2   # DTensor-based FSDP2; 1 = legacy FlatParameter
      fsdp_reshard_after_forward: true   # FSDP2's ZeRO-3 knob (replaces sharding_strategy)
      fsdp_auto_wrap_policy: TRANSFORMER_BASED_WRAP
      fsdp_transformer_layer_cls_to_wrap: {LAYER_CLS}   # <- one shard unit per decoder block
      fsdp_offload_params: false    # true => CPUOffloadPolicy
      fsdp_backward_prefetch: BACKWARD_PRE    # overlap the next all-gather with this backward
      fsdp_forward_prefetch: false
      fsdp_use_orig_params: true    # keeps param identity: needed for LoRA / param groups
      fsdp_sync_module_states: true    # rank 0 broadcasts init to all ranks
      fsdp_cpu_ram_efficient_loading: true    # only rank 0 loads the checkpoint into RAM
      fsdp_state_dict_type: SHARDED_STATE_DICT    # FULL_STATE_DICT can OOM rank 0 on a big model
      fsdp_activation_checkpointing: true
    """).lstrip(), encoding="utf-8")

print("wrote acc_deepspeed.yaml and acc_fsdp.yaml\n")
print("# the shipping form — identical training code, config chooses the backend:")
print("accelerate launch --config_file acc_fsdp.yaml       train_hf_trainer.py")
print("accelerate launch --config_file acc_deepspeed.yaml  train_hf_trainer.py")
print((WORKDIR / "acc_fsdp.yaml").read_text())

In [ ]:
# ---- Run the matrix ---------------------------------------------------------------------
NPROC = max(N_GPU, 1)
RESULTS_FILE = WORKDIR / "shard_results.jsonl"
RESULTS_FILE.unlink(missing_ok=True)

ARMS = [
    ("DDP (no sharding)", ["--backend", "ddp"]),
    ("ZeRO-1", ["--backend", "deepspeed", "--ds-config", "ds_config_stage1.json"]),
    ("ZeRO-2", ["--backend", "deepspeed", "--ds-config", "ds_config_stage2.json"]),
    ("ZeRO-3", ["--backend", "deepspeed", "--ds-config", "ds_config_stage3.json"]),
    ("ZeRO-2 + CPU offload", ["--backend", "deepspeed", "--ds-config", "ds_config_stage2_offload.json"]),
    ("ZeRO-3 + CPU offload", ["--backend", "deepspeed", "--ds-config", "ds_config_stage3_offload.json"]),
    ("FSDP2 FULL_SHARD", ["--backend", "fsdp"]),
    ("FSDP2 SHARD_GRAD_OP", ["--backend", "fsdp", "--reshard-after-forward", "false"]),
    ("FSDP2 + CPU offload", ["--backend", "fsdp", "--fsdp-offload"]),
]

# ZeRO-3/FSDP gather-then-free every layer, every step — the textbook allocator-fragmentation
# workload. expandable_segments lets the caching allocator grow segments instead of stranding
# them, and it is the difference between "OOM at step 40" and "runs".
ENV = {**os.environ, "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
       "TOKENIZERS_PARALLELISM": "false"}

def classify_failure(out, err, code):
    """An OOM is a RESULT here, not an error — it is the wall this technique exists to move."""
    blob = out + err
    if "OutOfMemoryError" in blob or "CUDA out of memory" in blob:
        return "CUDA OOM"
    if code == -9 or "Signal 9 (SIGKILL)" in blob:
        return "host RAM OOM (SIGKILL)"
    if "Timed out" in blob or "NCCL timeout" in blob:
        return "collective timeout (check drop_last / NCCL_SOCKET_IFNAME)"
    return f"failed (exit {code})"

def run_arm(tag, extra):
    cmd = ["torchrun", "--standalone", f"--nproc_per_node={NPROC}", "train_shard.py",
           "--data", DATA_DIR, "--model", MODEL_ID, "--tag", tag,
           "--bs", str(MICRO_BS), "--seq", str(SEQ_LEN),
           "--pin-memory", str(PIN_MEMORY).lower(),
           "--steps", "12", "--warmup", "3", "--out", str(RESULTS_FILE)] + extra
    print(f"\n$ {' '.join(cmd)}")
    proc = subprocess.run(cmd, capture_output=True, text=True, cwd=WORKDIR, env=ENV)
    if "SHARD_RESULT" not in proc.stdout:
        reason = classify_failure(proc.stdout, proc.stderr, proc.returncode)
        print(f"  -> {reason}")
        if reason.startswith("failed"):        # unexpected: show the tail so it is debuggable
            print(proc.stdout[-2000:]); print(proc.stderr[-2000:])
        # Keep the row. "This configuration does not fit" is the single most useful data
        # point in a sharding study, and dropping it would hide the actual finding.
        return {"tag": tag, "failed": reason}
    rec = json.loads(proc.stdout.split("SHARD_RESULT ", 1)[1].splitlines()[0])
    print(f"  -> {rec['tokens_per_sec_per_gpu']:,.0f} tok/s/gpu | "
          f"{rec['peak_reserved_gb']:.2f} GB peak | loss {rec['last_loss']:.4f}")
    return rec

results = {tag: run_arm(tag, extra) for tag, extra in ARMS}
ok = {k: v for k, v in results.items() if "failed" not in v}
print(f"\n{len(ok)}/{len(ARMS)} arms completed at world_size={NPROC}")
for k, v in results.items():
    if "failed" in v:
        print(f"  did not fit: {k:<24} {v['failed']}")

### Step 6 — Results

Read the table with the two axes separate, and read the `world_size` first:

- **At `world_size = 1`** the sharding rows are *expected* to be identical to DDP. That is not a null result — it is the model being confirmed. Only the **offload** rows should move, and they should move a lot on memory and badly on throughput.
- **At `world_size ≥ 2`** the ZeRO/FSDP rows should fall roughly as `1/P` on the terms each stage partitions, and ZeRO-3/`FULL_SHARD` should give up throughput in proportion to how bad the interconnect is.
- **`last_loss` must agree across every arm.** Sharding is numerically neutral. Disagreement means a wrong `reduce_dtype`, a desynced sampler, or an accumulation-boundary bug — never "a distributed effect."

In [ ]:
base = ok.get("DDP (no sharding)")
print(f"world_size = {NPROC}   |   {GPU_NAME}   |   {MODEL_ID}   |   dtype = {DTYPE}\n")
print(f"{'arm':<24}{'GB peak':>9}{'vs DDP':>9}{'tok/s/gpu':>11}{'vs DDP':>9}"
      f"{'comm':>7}{'loss':>9}")
print("-" * 78)
for tag, r in results.items():
    if "failed" in r:
        print(f"{tag:<24}{r['failed']:>45}")
        continue
    mem_r = f"{r['peak_reserved_gb']/base['peak_reserved_gb']:.2f}x" if base else "—"
    thr_r = f"{r['tokens_per_sec_per_gpu']/base['tokens_per_sec_per_gpu']:.2f}x" if base else "—"
    print(f"{tag:<24}{r['peak_reserved_gb']:>9.2f}{mem_r:>9}"
          f"{r['tokens_per_sec_per_gpu']:>11,.0f}{thr_r:>9}"
          f"{r['comm_factor_vs_ddp']:>6.1f}x{r['last_loss']:>9.4f}")

losses = [r["last_loss"] for r in ok.values()]
if losses:
    print(f"\nloss spread across {len(losses)} completed arms: {max(losses)-min(losses):.4f}"
          f"   (should be ~0 — sharding is numerically neutral)")
else:
    print("\nNo arm completed. Every row above is a memory verdict, not a bug: read the reason,")
    print("then lower MICRO_BS/SEQ_LEN in Step 2 (the loss term is cubic in none of them but")
    print("linear in both) and re-run. The budget printout in Step 2 predicts this in advance.")

if NPROC == 1:
    print("\nworld_size=1: the ZeRO/FSDP rows matching DDP is the CORRECT result, not a failure.")
    print("The sharding factor is 1. What moved is the OFFLOAD rows — a different mechanism.")
    print("Step 1's table has the sharding numbers you would get on real multi-GPU hardware.")

In [ ]:
# ---- Plot: two measures of different scale => two panels, never a dual axis --------------
# Colour encodes the FRAMEWORK (the entity), not the rank of the bar. The baseline is a
# neutral reference grey rather than a categorical hue: it is the thing being compared against.
import matplotlib.pyplot as plt

SURFACE, INK, INK_MUTED, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e6e5e1"
FRAMEWORK = {"baseline": "#6f6e6a", "DeepSpeed": "#2a78d6", "FSDP": "#eb6834"}

def family(tag):
    return "DeepSpeed" if tag.startswith("ZeRO") else ("FSDP" if tag.startswith("FSDP") else "baseline")

tags = list(ok)          # only completed arms are plottable; failures live in the table
if not tags:
    raise SystemExit("no completed arms to plot — see the table above for why")
colors = [FRAMEWORK[family(t)] for t in tags]
panels = [
    ("Peak VRAM per GPU", "GB reserved",
     [ok[t]["peak_reserved_gb"] for t in tags], "lower is better"),
    ("Throughput", "tokens / sec / GPU",
     [ok[t]["tokens_per_sec_per_gpu"] for t in tags], "higher is better"),
]

fig, axes = plt.subplots(1, 2, figsize=(12, 0.42 * len(tags) + 2.2), facecolor=SURFACE)
for ax, (title, unit, vals, hint) in zip(axes, panels):
    ax.set_facecolor(SURFACE)
    y = range(len(tags))
    bars = ax.barh(y, vals, height=0.62, color=colors)
    for b, v in zip(bars, vals):
        ax.text(b.get_width() * 1.02, b.get_y() + b.get_height() / 2,
                f"{v:,.0f}" if v > 100 else f"{v:.2f}",
                va="center", ha="left", color=INK, fontsize=9)
    ax.set_yticks(list(y)); ax.set_yticklabels(tags, fontsize=9, color=INK)
    ax.invert_yaxis()
    ax.set_xlim(0, max(vals) * 1.25)
    ax.set_title(f"{title}   ·   {unit}", loc="left", color=INK, fontsize=11, pad=10)
    ax.text(0, 1.0, hint, transform=ax.transAxes, ha="left", va="bottom",
            color=INK_MUTED, fontsize=9)
    ax.grid(axis="x", color=GRID, linewidth=0.8); ax.set_axisbelow(True)
    for s in ("top", "right", "left"):
        ax.spines[s].set_visible(False)
    ax.spines["bottom"].set_color("#d8d7d2")
    ax.tick_params(colors=INK_MUTED, labelsize=9, length=0)

# The physical ceiling this whole technique exists to get past.
axes[0].axvline(GPU_GB, color="#e34948", linewidth=1.4, linestyle=(0, (4, 3)))
axes[0].text(GPU_GB, -0.9, f" {GPU_GB:.0f} GB card", color="#e34948", fontsize=9, va="bottom")

handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in FRAMEWORK.values()]
fig.legend(handles, list(FRAMEWORK), loc="lower center", ncol=3, frameon=False,
           bbox_to_anchor=(0.5, -0.06), fontsize=10, labelcolor=INK)
fig.suptitle(f"ZeRO vs FSDP — {MODEL_ID.split('/')[-1]} full fine-tune, "
             f"world_size={NPROC} on {GPU_NAME}",
             x=0.008, ha="left", color=INK, fontsize=12.5, fontweight="semibold")
fig.tight_layout(rect=(0, 0.03, 1, 0.93))
plt.savefig("sharding_benchmark.png", dpi=160, bbox_inches="tight", facecolor=SURFACE)
plt.show()

### Step 7 — Checkpointing: the part that actually breaks in production

Once parameters are sharded, **there is no `model.state_dict()` that means what you think it means.** Every rank holds a slice. This is where most first distributed projects lose a weekend, so it belongs in the comparison rather than in a footnote.

**DeepSpeed ZeRO-3** — two paths:
- `stage3_gather_16bit_weights_on_model_save: true` (set in Step 3) makes `engine.save_16bit_model(dir)` gather the shards and write a normal checkpoint. Convenient; **can OOM rank 0** for a large model, since one rank briefly holds everything.
- Otherwise `engine.save_checkpoint(dir)` writes shards plus a `zero_to_fp32.py` script; run it offline (`python zero_to_fp32.py <ckpt_dir> <out.bin>`) on a big-RAM CPU box. This is the safe production path.

**FSDP** — use the parallelism-agnostic API:
- `torch.distributed.checkpoint.state_dict.get_state_dict(model, optimizer)` returns full or sharded state depending on `StateDictOptions`, and works identically for FSDP1, FSDP2 and DDP. The older `FSDP.state_dict_type(...)` context managers are the deprecated route.
- `torch.distributed.checkpoint.save/load` (**DCP**) writes a sharded checkpoint that can be **loaded back at a different `world_size`** — the capability that matters when your allocation changes from 8 GPUs to 32, and something ZeRO handles less gracefully.

Two rules that survive both frameworks:

1. **`FULL_STATE_DICT` on a large model is an OOM waiting to happen.** Default to sharded, consolidate offline.
2. **Save the optimizer state, not just the weights.** A CPT/pretraining run resumed without Adam moments restarts from a cold optimizer and the loss visibly spikes.

In [ ]:
ckpt_demo = r"""
# --- DeepSpeed --------------------------------------------------------------------------
# Sharded (safe, resumable, needs offline consolidation):
engine.save_checkpoint("./ckpt_ds")                 # per-rank shards + zero_to_fp32.py
#   $ python ./ckpt_ds/zero_to_fp32.py ./ckpt_ds ./pytorch_model.bin

# Consolidated in-process (convenient, RISKS OOM on rank 0 for a large model);
# requires stage3_gather_16bit_weights_on_model_save: true in the config.
engine.save_16bit_model("./ckpt_ds_16bit")

# --- FSDP (parallelism-agnostic API — works for FSDP1, FSDP2 and DDP alike) -------------
import torch.distributed.checkpoint as dcp
from torch.distributed.checkpoint.state_dict import (
    StateDictOptions, get_state_dict, set_state_dict,
)

# Sharded save: every rank writes its slice. Reloadable at a DIFFERENT world_size.
msd, osd = get_state_dict(model, opt)
dcp.save({"model": msd, "optim": osd}, checkpoint_id="./ckpt_fsdp")

# Resume (possibly on a different number of GPUs):
msd, osd = get_state_dict(model, opt)
dcp.load({"model": msd, "optim": osd}, checkpoint_id="./ckpt_fsdp")
set_state_dict(model, opt, model_state_dict=msd, optim_state_dict=osd)

# Consolidated export for serving — rank 0 only, and only when the model fits in host RAM.
full, _ = get_state_dict(model, opt, options=StateDictOptions(full_state_dict=True, cpu_offload=True))
if int(os.environ.get("RANK", 0)) == 0:
    torch.save(full, "pytorch_model.bin")
"""
Path("checkpointing_reference.py").write_text(ckpt_demo, encoding="utf-8")
print(ckpt_demo)

---

## **[Key Observations]**

*Fill in from the runs above. A sharding result without its `world_size` and interconnect is meaningless — copy the configuration row every time you quote a number.*

### Run configuration

| Setting | Value |
|---|---|
| GPUs × model × VRAM each | |
| `world_size` actually used | |
| Interconnect (NVLink / PCIe / IB) — from `nvidia-smi topo -m` | |
| Host RAM (and whether offload pinned it) | |
| Model / params / vocab (probe-selected) | |
| Model state GB vs. loss-term GB (Step 2 budget) | |
| dtype (bf16 mixed vs fp32) | |
| `micro_bs` × `accum` × `world_size` × `seq` = effective tokens/step | |

### Results

| Arm | Peak GB/GPU | vs DDP | tok/s/GPU | vs DDP | comm × DDP | final loss |
|---|---|---|---|---|---|---|
| DDP (no sharding) | | 1.00× | | 1.00× | 1.0× | |
| ZeRO-1 | | | | | 1.0× | |
| ZeRO-2 | | | | | 1.0× | |
| ZeRO-3 | | | | | 1.5× | |
| ZeRO-2 + CPU offload | | | | | 1.0× | |
| ZeRO-3 + CPU offload | | | | | 1.5× | |
| FSDP2 `FULL_SHARD` | | | | | 1.5× | |
| FSDP2 `SHARD_GRAD_OP` | | | | | 1.0× | |
| FSDP2 + CPU offload | | | | | 1.5× | |

### The two axes, kept separate

| Question | Answer from this run |
|---|---|
| At this `world_size`, how much did **sharding** save? | |
| How much did **offload** save, independent of `world_size`? | |
| Did ZeRO-3 and FSDP2 `FULL_SHARD` land within noise of each other? *(they implement the same algorithm — a large gap is a config difference, not a framework difference)* | |
| Was the throughput cost of ZeRO-3 close to the `1.5×` comm prediction, or much worse? *(much worse ⇒ interconnect-bound)* | |
| Loss spread across all arms | should be ~0 |

### Things worth logging every time

- **`max_memory_reserved` reduced with `MAX` across ranks**, not rank 0's allocated. The fattest rank OOMs the job.
- **Scaling efficiency** `throughput(P) / (P × throughput(1))`. Below ~0.7, fix the stage or the interconnect before buying GPUs.
- **Time-to-first-step.** ZeRO-3 `zero.Init` and FSDP meta-device materialisation cost real minutes and never appear in steady-state throughput.
- **Host RAM high-water mark whenever offload is on** — the second, invisible OOM.
- Whether `drop_last=True` is set on the sampler. A hang with no traceback is almost always this.

### Sensitivity sweeps worth running

- `world_size ∈ {1, 2, 4, 8}` for one stage — the scaling-efficiency curve is the single most useful plot in distributed training, and the only way to know where your interconnect gives up.
- **`reshard_after_forward` `True` vs `False`** on FSDP2 — the cleanest possible isolation of ZeRO-3's extra `0.5×` communication, since nothing else changes.
- **Sequence length `∈ {512, 1024, 4096}`** — ZeRO-3's relative cost should *fall* as `B·L` grows, because arithmetic intensity rises. If it does not, you are latency-bound on small collectives; revisit the wrap granularity.
- **Wrap granularity**: wrap every *other* decoder block instead of every block, and watch peak memory and throughput move in opposite directions.

## Export — Download the Configs, Scripts and Results (Optional)

> The deliverable of a sharding investigation is **not a checkpoint** — it is the configuration that makes the run possible, plus the evidence it works. Ship the configs, the launch scripts, the results JSONL and the topology, and the next person can reproduce the decision instead of re-running the search.

In [ ]:
import shutil

bundle = WORKDIR / "sharding_bundle"
bundle.mkdir(exist_ok=True)
for f in ["train_shard.py", "checkpointing_reference.py", "acc_fsdp.yaml", "acc_deepspeed.yaml",
          "shard_results.jsonl", "sharding_benchmark.png"] + [p.name for p in CONFIGS.values()]:
    if (WORKDIR / f).exists():
        shutil.copy(WORKDIR / f, bundle / f)

topo = subprocess.run(["nvidia-smi", "topo", "-m"], capture_output=True, text=True).stdout if N_GPU else "no GPU"
(bundle / "TOPOLOGY.txt").write_text(
    f"gpus={N_GPU} x {GPU_NAME} ({GPU_GB:.1f} GB, sm_{SM})\nhost_ram_gb={HOST_GB:.1f}\n"
    f"torch={torch.__version__}\ndtype={DTYPE}\n\n{topo}"
)

output_filename = "sharding_bundle.zip"
shutil.make_archive(output_filename.replace(".zip", ""), "zip", bundle)
print(f"File: {output_filename}  ({os.path.getsize(output_filename)/1e6:.2f} MB)")

### Download to your machine

In [ ]:
from google.colab import files
files.download(output_filename)

### Or back up to Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

destination_folder = "/content/drive/MyDrive/colab_models"
if os.path.exists(output_filename):
    os.makedirs(destination_folder, exist_ok=True)
    shutil.copy(output_filename, os.path.join(destination_folder, output_filename))
    print("Backed up to Drive:", destination_folder)
else:
    print("Error: zip not found — run the export cell first.")